# PA6: Naive Bayes and Measuring Performance
- Programmer: Lydia Lonzarich
- Class: CPSC 322-01, Fall 2025
- Programming Assignment #6
- Date of current version: 11/17/2025
- Description: this notebook performs Naive Bayes classification and measures performance using precision, recall, and the F1 score.

In [87]:
# some useful mysklearn package import statements and reloads
import importlib

import numpy as np

import mysklearn.myutils
importlib.reload(mysklearn.myutils)
# import mysklearn.myutils as myutils
from mysklearn.myutils import cross_val_predict

import mysklearn.mypytable
importlib.reload(mysklearn.mypytable)
from mysklearn.mypytable import MyPyTable 

import mysklearn.myclassifiers
importlib.reload(mysklearn.myclassifiers)
from mysklearn.myclassifiers import MyDummyClassifier, MyNaiveBayesClassifier

import mysklearn.myevaluation
importlib.reload(mysklearn.myevaluation)
# import mysklearn.myevaluation as myevaluation
from mysklearn.myevaluation import binary_precision_score, binary_recall_score, confusion_matrix

from tabulate import tabulate

# Load the dataset
- In this step, I load the titanic.csv dataset.
- I create a MyPyTable object to store the data in. 

In [88]:
filename = "input_data/titanic.csv"
pytable = MyPyTable()
pytable.load_from_file(filename)

# convert all integer values to floats.
pytable.convert_to_numeric()

# convert the dataset to an array.
data = np.array(pytable.data, dtype=object)

# Classify using Naive Bayes
- In this step, I use my Naive Bayes classifier and k-fold cross validation (with k=10) to predict whether passengers aboard the titanic ship will survive the tragic sinking.
- This is a binary classification task, meaning my classifier will predict 'yes' or 'no' based on passenger class, age, and sex attributes.
- I assume that the Naive Bayes classifier will perform much better than the dummy classifier because Naive Bayes is an eager learning classifier. This means the model learns generalizable patterns in the training data before it generates predictions for unseen instances, which results in reduced prediction speed.

In [89]:
# "get" X (data) and y (corresponding labels) out of the pytable.
# find indices of the 'cylinders', 'weight', 'acceleration', and 'mpg' column in the table.
class_indices = pytable.column_names.index("class")
age_indices = pytable.column_names.index("age")
sex_indices = pytable.column_names.index("sex")
survived_indices = pytable.column_names.index("survived")

# separate data into X (samples) and y (corresponding labels).
X = np.column_stack((data[:, class_indices], data[:, age_indices], data[:, sex_indices]))
y = data[:, survived_indices]

# compute the avg acc and error rate, avg precision, avg recall, and avg F1 over each train/test split of the data.
nb_acc, nb_err_rate, nb_precision, nb_recall, nb_f1, nb_y_trues, nb_y_preds = cross_val_predict(X, y, 10, MyNaiveBayesClassifier, True)

print("Naive Bayes Classifier Results:")
print("accuracy = ", nb_acc)
print("error rate = ", nb_err_rate)
print("precision = ", nb_precision)
print("recall: ", nb_recall)
print("F1-score: ", nb_f1)


Naive Bayes Classifier Results:
accuracy =  0.7791875771287536
error rate =  0.2208124228712464
precision =  0.7916067379206189
recall:  0.9154362416107382
F1-score:  0.8487723289308736


# Classify Using Dummy Classifier
- In this step, I use my Dummy classifier and k-fold cross validation (with k=10) to predict whether passengers aboard the titanic ship will survive the tragic sinking.
- This is a binary classification task, meaning my classifier will predict 'yes' or 'no' based on passenger class, age, and sex attributes.

In [90]:
# "get" X (data) and y (corresponding labels) out of the pytable.
# find indices of the 'cylinders', 'weight', 'acceleration', and 'mpg' column in the table.
class_indices = pytable.column_names.index("class")
age_indices = pytable.column_names.index("age")
sex_indices = pytable.column_names.index("sex")
survived_indices = pytable.column_names.index("survived")

# separate data into X (samples) and y (corresponding labels).
X = np.column_stack((data[:, class_indices], data[:, age_indices], data[:, sex_indices]))
y = data[:, survived_indices]

# compute the avg acc and error rate for each train/test split of the data.
dc_acc, dc_err_rate, dc_precision, dc_recall, dc_f1, dc_y_trues, dc_y_preds = cross_val_predict(X, y, 10, MyDummyClassifier, True)

print("Dummy Classifier Results:")
print("accuracy = ", dc_acc)
print("error rate = ", dc_err_rate)
print("Dummy Classifier: precision = ", dc_precision)
print("recall: ", dc_recall)
print("F1-score: ", dc_f1)

Dummy Classifier Results:
accuracy =  0.6769662690250925
error rate =  0.3230337309749075
Dummy Classifier: precision =  0.6769662690250925
recall:  1.0
F1-score:  0.8073698088332234


# Display Confusion Matrices 
- In this step, I display the confusion matrix to evaluate the performance of both the Naive Bayes and Dummy Classifiers on the titanic classification task.

In [91]:
print("=============================")
print("STEP 4: Confusion Matrices")
print("=============================")

labels = ["yes", "no"]

headers = ["class", "yes", "no", "Total", "Recognition (%)"]



print("Naive Bayes Classifier (Stratified 10-Fold Cross Validation Results):")
nb_matrix = confusion_matrix(nb_y_trues, nb_y_preds, labels) # ==> a list of lists.
nb_matrix = np.array(nb_matrix)
totals = nb_matrix.sum(axis=1) # get the totals for each class in the table.

# initialize a list to store the recognition % for each row. 
nb_recognition = [] 

# iterate through each row in the nb_matrix to calculate its recognition %.
# recognition: - another measure of how well the model predicted the true label. 
#              - we look at how close the diagonal values in the cm (row i, column i) is to the the total for a row i. 
for row in range(len(nb_matrix)):
    # if there are instances the current row, calcuate its recognition %: diagonal / row_total * 100 
    if totals[row] > 0:
        rec = nb_matrix[row, row] / totals[row] * 100
    else:
        rec = 0

    nb_recognition.append(rec)

# add the total counts and recognition (%) of each row to cm matrix. 
completed_nb_cm = []
for i, label in enumerate(labels):
    # for each row, append the data as: mpg ranking label | the actual data for each ranking | total count for the row | recognition percept for the row
    completed_nb_cm.append([label] + list(nb_matrix[i]) + [totals[i], round(nb_recognition[i], 1)])

# create the formatted cm. 
nb_cm_table = tabulate(completed_nb_cm, headers=headers, tablefmt="grid")
print(nb_cm_table)



print("-------------------------------------------------------------------")
print("Dummy Classifier (Stratified 10-Fold Cross Validation Results):")
dummy_matrix = confusion_matrix(dc_y_trues, dc_y_preds, labels)
dummy_matrix = np.array(dummy_matrix)
totals = dummy_matrix.sum(axis=1) # get the totals for each class in the table.

# initialize a list to store the recognition % for each row. 
dummy_recognition = [] 

# iterate through each row in the knn_matrix to calculate its recognition %.
for row in range(len(dummy_matrix)):
    # if there are instances the current row, calcuate its recognition %: diagonal / row_total * 100 
    if totals[row] > 0:
        rec = dummy_matrix[row, row] / totals[row] * 100
    else:
        rec = 0

    dummy_recognition.append(rec)

# add the total counts and recognition (%) of each row to cm matrix. 
completed_dummy_cm = []
for i, label in enumerate(labels):
    # for each row, append the data as: mpg ranking label | the actual data for each ranking | total count for the row | recognition percept for the row
    completed_dummy_cm.append([label] + list(dummy_matrix[i]) + [totals[i], round(dummy_recognition[i], 1)])

# create the formatted cm. 
dummy_cm_table = tabulate(completed_dummy_cm, headers=headers, tablefmt="grid")
print(dummy_cm_table)


STEP 4: Confusion Matrices
Naive Bayes Classifier (Stratified 10-Fold Cross Validation Results):
+---------+-------+------+---------+-------------------+
| class   |   yes |   no |   Total |   Recognition (%) |
+=========+=======+======+=========+===================+
| yes     |  1364 |  126 |    1490 |              91.5 |
+---------+-------+------+---------+-------------------+
| no      |   360 |  351 |     711 |              49.4 |
+---------+-------+------+---------+-------------------+
-------------------------------------------------------------------
Dummy Classifier (Stratified 10-Fold Cross Validation Results):
+---------+-------+------+---------+-------------------+
| class   |   yes |   no |   Total |   Recognition (%) |
+=========+=======+======+=========+===================+
| yes     |  1490 |    0 |    1490 |               100 |
+---------+-------+------+---------+-------------------+
| no      |   711 |    0 |     711 |                 0 |
+---------+-------+------+----

# Mini Reflection
Overall, my assumptions align with the results - that the Naive Bayes classifier outperforms the Dummy classifier. This is most likely because the Naive Bayes classifier is learning generalizable patterns in the training data that help it make informed predictions on unseen instances. On the other hand, the dummy classifier has no "skill" or logical reason for choosing one class label prediction over the other for each unseen instance. 